# Tarea B — Temperatura, top-p y entropía de la distribución de salida

**MMIA 6013 · Tarea de práctica para el solver del Taller 03 v2**

Entrega: un notebook de Jupyter (.ipynb) ejecutado de principio a fin, sin errores y con las salidas visibles.

## Preámbulo

Un modelo de lenguaje produce, para el siguiente token, un vector de logits. En esta tarea se trabaja con un vocabulario de seis tokens, t0 a t5, con los siguientes logits (datos de entrada del enunciado):

| Token | t0 | t1 | t2 | t3 | t4 | t5 |
|---|---|---|---|---|---|---|
| Logit | 2.0 | 1.0 | 0.5 | 0.2 | -1.0 | -3.0 |

El notebook recorre cuatro partes:

1. **Parte 1 — Softmax con temperatura:** implementación numéricamente estable en NumPy de p_i = exp(z_i/T) / Σ_j exp(z_j/T); se calculan las distribuciones para T = 0.5, T = 1 y T = 2 y la entropía en bits de cada una, presentadas en una tabla con cuatro decimales.
2. **Parte 2 — Muestreo de núcleo (top-p):** sobre la distribución de T = 1 se aplica top-p con p = 0.9, se conserva el conjunto mínimo de tokens cuya probabilidad acumulada alcanza el umbral y se renormaliza.
3. **Parte 3 — Comprobación empírica:** se toman 10 000 muestras de la distribución renormalizada con `numpy.random.default_rng(0)`, se comparan las frecuencias observadas con las probabilidades teóricas en un gráfico de barras y se calcula la divergencia KL en bits.
4. **Parte 4 — Pregunta conceptual:** análisis de los límites T → 0 y T → ∞ y de por qué T → 0 equivale a la decodificación voraz (greedy), usando las entropías de la Parte 1 como evidencia.

Todas las cifras se calculan en las celdas de código y se reportan como salidas ejecutadas; ninguna se escribe a mano.

## Parte 1 — Softmax con temperatura

Se implementa en NumPy la softmax con temperatura, p_i = exp(z_i/T) / Σ_j exp(z_j/T), de forma numéricamente estable: antes de exponenciar se resta el máximo de los logits, es decir se evalúa exp((z_i − max_j z_j)/T). Este desplazamiento no altera el resultado (la softmax es invariante ante desplazamientos constantes del logit) pero evita desbordamientos numéricos. La entropía se calcula en bits como H = −Σ p_i log2 p_i. La salida presenta, con cuatro decimales, la distribución completa para T = 0.5, T = 1 y T = 2, la entropía de cada una y la verificación de que cada fila sume 1.

In [1]:
import json
import numpy as np


def softmax_with_temperature(z, T):
    """Softmax con temperatura, numéricamente estable.

    p_i = exp(z_i / T) / sum_j exp(z_j / T)
    Estabilidad: se resta el máximo de los logits escalados antes de
    exponenciar, evitando desbordamiento de exp.
    """
    z = np.asarray(z, dtype=float)
    scaled = z / float(T)
    scaled = scaled - np.max(scaled)  # resta del máximo (estabilidad numérica)
    exp_scaled = np.exp(scaled)
    return exp_scaled / np.sum(exp_scaled)


def entropy_bits(p):
    """Entropía en bits: H = -sum p_i log2 p_i, con la convención 0*log2(0)=0."""
    p = np.asarray(p, dtype=float)
    mask = p > 0
    return float(-np.sum(p[mask] * np.log2(p[mask])))


# ---- Datos de la tarea: vocabulario de seis tokens y sus logits ----
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]
temperatures = [0.5, 1.0, 2.0]

z = np.array(logits, dtype=float)

distributions = {}
entropies = {}
sums = {}

for T in temperatures:
    p = softmax_with_temperature(z, T)
    distributions[T] = p
    entropies[T] = entropy_bits(p)
    sums[T] = float(np.sum(p))

# ---- Tabla con cuatro decimales (formato de cadena; pandas no está disponible) ----
header = (
    f"{'T':<5} | "
    + " | ".join(f"{t:>7}" for t in tokens)
    + f" | {'H (bits)':>9} | {'suma':>6}"
)
lines = ["Parte 1 — Softmax con temperatura (numéricamente estable)",
         f"Logits: {logits}",
         "",
         header,
         "-" * len(header)]
for T in temperatures:
    row = (
        f"{T:<5.1f} | "
        + " | ".join(f"{p:7.4f}" for p in distributions[T])
        + f" | {entropies[T]:9.4f} | {sums[T]:6.4f}"
    )
    lines.append(row)
table_str = "\n".join(lines)
print(table_str)
print()

# ---- Impresión explícita de las entropías ----
print("Entropías en bits (H = -sum p_i log2 p_i):")
for T in temperatures:
    print(f"  T = {T}: H = {entropies[T]:.4f} bits")
print()

# ---- Verificación de que cada distribución suma 1 ----
print("Verificación (cada fila de la tabla debe sumar 1.0000):")
for T in temperatures:
    print(f"  T = {T}: suma = {sums[T]:.10f}")
print()

# ---- Escritura de resultados a JSON ----
results = {
    "tokens": tokens,
    "logits": [float(v) for v in logits],
    "temperatures": [float(T) for T in temperatures],
    "distributions": {
        f"T={T}": {tok: float(p) for tok, p in zip(tokens, distributions[T])}
        for T in temperatures
    },
    "entropies_bits": {f"T={T}": float(entropies[T]) for T in temperatures},
    "distribution_sums": {f"T={T}": float(sums[T]) for T in temperatures},
}

with open("resultados.json", "w") as f:
    json.dump(results, f, indent=2)

print("Resultados escritos en resultados.json")


Parte 1 — Softmax con temperatura (numéricamente estable)
Logits: [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]

T     |      t0 |      t1 |      t2 |      t3 |      t4 |      t5 |  H (bits) |   suma
--------------------------------------------------------------------------------------
0.5   |  0.8231 |  0.1114 |  0.0410 |  0.0225 |  0.0020 |  0.0000 |    0.9147 | 1.0000
1.0   |  0.5516 |  0.2029 |  0.1231 |  0.0912 |  0.0275 |  0.0037 |    1.7998 | 1.0000
2.0   |  0.3583 |  0.2173 |  0.1693 |  0.1457 |  0.0800 |  0.0294 |    2.2888 | 1.0000

Entropías en bits (H = -sum p_i log2 p_i):
  T = 0.5: H = 0.9147 bits
  T = 1.0: H = 1.7998 bits
  T = 2.0: H = 2.2888 bits

Verificación (cada fila de la tabla debe sumar 1.0000):
  T = 0.5: suma = 1.0000000000
  T = 1.0: suma = 1.0000000000
  T = 2.0: suma = 1.0000000000

Resultados escritos en resultados.json


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T = 1 de la Parte 1 se aplica top-p (muestreo de núcleo) con p = 0.9: se ordenan los tokens de mayor a menor probabilidad, se conserva el conjunto mínimo cuya probabilidad acumulada alcanza o supera 0.9 y se renormalizan las probabilidades de los tokens supervivientes para que sumen 1. La salida muestra el orden con sus probabilidades acumuladas, qué tokens sobreviven al núcleo y cuáles se recortan, y la distribución renormalizada con cuatro decimales.

In [2]:
import json
import numpy as np

# ------------------------------------------------------------------
# Subtarea s2 (Parte 2): muestreo de núcleo (top-p) con p = 0.9
# sobre la distribución softmax con T = 1 de la Tarea B (MMIA 6013).
# Se recomputa la softmax de T = 1 (Parte 1) a partir de los logits.
# ------------------------------------------------------------------

# Datos literales del enunciado
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
T = 1.0
p_threshold = 0.9

# ------------------------------------------------------------------
# Paso 1: softmax con temperatura, numéricamente estable
# p_i = exp((z_i - max)/T) / sum_j exp((z_j - max)/T)
# ------------------------------------------------------------------
def softmax_with_temperature(z, temperature):
    z = np.asarray(z, dtype=float)
    z_shift = (z - np.max(z)) / temperature
    e = np.exp(z_shift)
    return e / np.sum(e)

probs = softmax_with_temperature(logits, T)

# ------------------------------------------------------------------
# Paso 2: ordenar los tokens por probabilidad descendente
# ------------------------------------------------------------------
order = np.argsort(-probs, kind="stable")
sorted_tokens = [tokens[i] for i in order]
sorted_probs = probs[order]
cumulative = np.cumsum(sorted_probs)

# ------------------------------------------------------------------
# Paso 3: conjunto mínimo de tokens cuya probabilidad acumulada
# alcanza o supera 0.9
# ------------------------------------------------------------------
k = 0
cum_nucleus = 0.0
for cp in cumulative:
    k += 1
    cum_nucleus = float(cp)
    if cum_nucleus >= p_threshold:
        break

nucleus_pos = order[:k]
survivors = [tokens[i] for i in nucleus_pos]
excluded = [tokens[i] for i in order[k:]]

# ------------------------------------------------------------------
# Paso 4: renormalizar las probabilidades del núcleo para que sumen 1
# ------------------------------------------------------------------
renorm = probs[nucleus_pos] / cum_nucleus

# ------------------------------------------------------------------
# Resultados -> resultados.json
# ------------------------------------------------------------------
results = {
    "tokens": tokens,
    "logits": [float(v) for v in logits],
    "temperature": float(T),
    "p_threshold": float(p_threshold),
    "softmax_T1_full": {tok: float(pr) for tok, pr in zip(tokens, probs)},
    "softmax_T1_4dp": {tok: round(float(pr), 4) for tok, pr in zip(tokens, probs)},
    "sorted_order_desc": sorted_tokens,
    "sorted_probs_desc_4dp": [round(float(v), 4) for v in sorted_probs],
    "cumulative_probs_desc_4dp": [round(float(v), 4) for v in cumulative],
    "nucleus_size": int(k),
    "surviving_tokens": survivors,
    "excluded_tokens": excluded,
    "surviving_original_probs_4dp": {
        tok: round(float(probs[tokens.index(tok)]), 4) for tok in survivors
    },
    "nucleus_cumulative_prob": cum_nucleus,
    "nucleus_cumulative_prob_4dp": round(cum_nucleus, 4),
    "renormalized_distribution_full": {
        tok: float(pr) for tok, pr in zip(survivors, renorm)
    },
    "renormalized_distribution_4dp": {
        tok: round(float(pr), 4) for tok, pr in zip(survivors, renorm)
    },
    "renormalized_sum": float(np.sum(renorm)),
}

with open("resultados.json", "w") as f:
    json.dump(results, f, indent=2)

# ------------------------------------------------------------------
# Salida a stdout (los mismos números)
# ------------------------------------------------------------------
print("Logits:", {tok: float(v) for tok, v in zip(tokens, logits)})
print(f"\nSoftmax con T = {T} (numéricamente estable):")
for tok, pr in zip(tokens, probs):
    print(f"  {tok}: {pr:.4f}")
print(f"  suma = {np.sum(probs):.4f}")

print("\nTokens ordenados por probabilidad descendente:")
for tok, pr, cp in zip(sorted_tokens, sorted_probs, cumulative):
    print(f"  {tok}: p = {pr:.4f}  acumulada = {cp:.4f}")

print(f"\nTop-p con p = {p_threshold}:")
print(f"  Tokens sobrevivientes (núcleo, k = {k}): {survivors}")
print(f"  Tokens recortados: {excluded}")
print(f"  Probabilidad acumulada del núcleo: {cum_nucleus:.4f}")

print("\nDistribución renormalizada del núcleo:")
for tok, pr in zip(survivors, renorm):
    orig = probs[tokens.index(tok)]
    print(f"  {tok}: original = {orig:.4f}  ->  renormalizada = {pr:.4f}")
print(f"  suma renormalizada = {np.sum(renorm):.4f}")


Logits: {'t0': 2.0, 't1': 1.0, 't2': 0.5, 't3': 0.2, 't4': -1.0, 't5': -3.0}

Softmax con T = 1.0 (numéricamente estable):
  t0: 0.5516
  t1: 0.2029
  t2: 0.1231
  t3: 0.0912
  t4: 0.0275
  t5: 0.0037
  suma = 1.0000

Tokens ordenados por probabilidad descendente:
  t0: p = 0.5516  acumulada = 0.5516
  t1: p = 0.2029  acumulada = 0.7546
  t2: p = 0.1231  acumulada = 0.8776
  t3: p = 0.0912  acumulada = 0.9688
  t4: p = 0.0275  acumulada = 0.9963
  t5: p = 0.0037  acumulada = 1.0000

Top-p con p = 0.9:
  Tokens sobrevivientes (núcleo, k = 4): ['t0', 't1', 't2', 't3']
  Tokens recortados: ['t4', 't5']
  Probabilidad acumulada del núcleo: 0.9688

Distribución renormalizada del núcleo:
  t0: original = 0.5516  ->  renormalizada = 0.5694
  t1: original = 0.2029  ->  renormalizada = 0.2095
  t2: original = 0.1231  ->  renormalizada = 0.1270
  t3: original = 0.0912  ->  renormalizada = 0.0941
  suma renormalizada = 1.0000


## Parte 3 — Comprobación empírica

Se toman 10 000 muestras de la distribución renormalizada de la Parte 2 con `numpy.random.default_rng(0)`. El gráfico de barras agrupadas compara, para cada token del núcleo, la frecuencia relativa observada con la probabilidad teórica. Además se calcula la divergencia KL en bits de la distribución empírica respecto de la teórica: KL(empírica ‖ teórica) = Σ p_teorica · log2(p_teorica / p_observada). Como muestra la salida, las frecuencias observadas están muy cerca de las teóricas y la KL es prácticamente 0, lo que confirma que el muestreo reproduce la distribución renormalizada dentro del error esperado por azar con ese tamaño de muestra.

In [3]:
# -*- coding: utf-8 -*-
"""
Subtask s3 (Parte 3 — Comprobación empírica).

Recomputa la distribución renormalizada top-p de la Parte 2
(softmax numéricamente estable con T = 1 sobre los logits del enunciado,
top-p con p = 0.9: conjunto mínimo con probabilidad acumulada >= 0.9,
renormalizado), toma exactamente 10 000 muestras con
numpy.random.default_rng(0), compara las frecuencias observadas con las
probabilidades teóricas en un gráfico de barras agrupadas y calcula la
divergencia KL en bits de la distribución empírica respecto de la teórica
sobre los tokens del núcleo.

Resultados -> resultados.json
Figura      -> parte3_comparacion_barras.png
"""
import json

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# Datos del enunciado (embedidos literalmente)
# ----------------------------------------------------------------------
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
tokens = np.array(["t0", "t1", "t2", "t3", "t4", "t5"])
T = 1.0            # temperatura de la Parte 1
p_threshold = 0.9  # umbral top-p de la Parte 2
n_samples = 10000  # número exacto de muestras de la Parte 3
seed = 0           # semilla exigida: numpy.random.default_rng(0)


# ----------------------------------------------------------------------
# Parte 1 (recomputada): softmax con temperatura, numéricamente estable
# ----------------------------------------------------------------------
def softmax_with_temperature(z, temp):
    z = np.asarray(z, dtype=float)
    z_shift = (z - np.max(z)) / temp  # resta del máximo -> estabilidad numérica
    e = np.exp(z_shift)
    return e / np.sum(e)


probs_T1 = softmax_with_temperature(logits, T)

# ----------------------------------------------------------------------
# Parte 2 (recomputada): top-p con p = 0.9 y renormalización
# ----------------------------------------------------------------------
order_desc = np.argsort(-probs_T1, kind="stable")
probs_sorted = probs_T1[order_desc]
cumulative = np.cumsum(probs_sorted)

# conjunto MÍNIMO de tokens con probabilidad acumulada >= p
k = int(np.searchsorted(cumulative, p_threshold, side="left")) + 1
k = min(k, len(tokens))

nucleus_idx = np.sort(order_desc[:k])  # núcleo en orden original de tokens
nucleus_tokens = tokens[nucleus_idx]
nucleus_probs = probs_T1[nucleus_idx]
renorm = nucleus_probs / np.sum(nucleus_probs)  # distribución renormalizada

excluded_tokens = np.setdiff1d(tokens, nucleus_tokens)

# ----------------------------------------------------------------------
# Parte 3: 10 000 muestras con numpy.random.default_rng(0)
# ----------------------------------------------------------------------
rng = np.random.default_rng(seed)
samples = rng.choice(nucleus_tokens, size=n_samples, p=renorm)

unique_labels, counts_int = np.unique(samples, return_counts=True)
count_map = {str(t): int(c) for t, c in zip(unique_labels, counts_int)}
counts = np.array([count_map.get(str(t), 0) for t in nucleus_tokens], dtype=float)
obs_freq = counts / float(n_samples)

# ----------------------------------------------------------------------
# Divergencia KL en bits, fórmula del enunciado:
# KL = sum p_teorica * log2(p_teorica / p_observada) sobre el núcleo
# (con 10 000 muestras y p_min ~ 0.094 todos los tokens del núcleo
#  aparecen; se excluyen términos con frecuencia observada nula por seguridad)
# ----------------------------------------------------------------------
mask = obs_freq > 0
kl_bits = float(np.sum(renorm[mask] * np.log2(renorm[mask] / obs_freq[mask])))

# ----------------------------------------------------------------------
# Gráfico de barras agrupadas: observado vs teórico
# ----------------------------------------------------------------------
x = np.arange(len(nucleus_tokens))
width = 0.38
fig, ax = plt.subplots(figsize=(8.5, 5.2))
bars_obs = ax.bar(x - width / 2.0, obs_freq, width,
                  label="Frecuencia relativa observada (10 000 muestras)",
                  color="#4C72B0", edgecolor="black", linewidth=0.5)
bars_teo = ax.bar(x + width / 2.0, renorm, width,
                  label="Probabilidad teórica (top-p renormalizada)",
                  color="#DD8452", edgecolor="black", linewidth=0.5)
for bars in (bars_obs, bars_teo):
    for b in bars:
        h = b.get_height()
        ax.annotate(f"{h:.4f}",
                    (b.get_x() + b.get_width() / 2.0, h),
                    ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(nucleus_tokens)
ax.set_xlabel("Token del núcleo (top-p, p = 0.9)")
ax.set_ylabel("Probabilidad")
ax.set_title("Parte 3 — Frecuencias observadas vs. probabilidad teórica\n"
             f"softmax T = 1, top-p p = 0.9, {n_samples} muestras, semilla {seed}")
ax.set_ylim(0.0, 1.15 * float(max(renorm.max(), obs_freq.max())))
ax.text(0.02, 0.97, f"Tokens excluidos por top-p: {', '.join(map(str, excluded_tokens))}",
        transform=ax.transAxes, fontsize=9, va="top", color="0.35")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig("parte3_comparacion_barras.png", dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------
# Resultados -> resultados.json (y stdout)
# ----------------------------------------------------------------------
results = {
    "logits": [float(v) for v in logits],
    "temperature": float(T),
    "p_threshold": float(p_threshold),
    "softmax_T1_full": {str(t): float(p) for t, p in zip(tokens, probs_T1)},
    "sorted_order_desc": [str(t) for t in tokens[order_desc]],
    "sorted_probs_desc": [float(v) for v in probs_sorted],
    "cumulative_probs_desc": [float(v) for v in cumulative],
    "nucleus_size": int(k),
    "surviving_tokens": [str(t) for t in nucleus_tokens],
    "excluded_tokens": [str(t) for t in excluded_tokens],
    "nucleus_cumulative_prob": float(cumulative[k - 1]),
    "renormalized_distribution": {str(t): float(p) for t, p in zip(nucleus_tokens, renorm)},
    "n_samples": int(n_samples),
    "rng_seed": int(seed),
    "observed_counts": {str(t): int(count_map.get(str(t), 0)) for t in nucleus_tokens},
    "observed_relative_frequencies": {str(t): float(f) for t, f in zip(nucleus_tokens, obs_freq)},
    "kl_empirical_vs_theoretical_bits": kl_bits,
    "kl_empirical_vs_theoretical_bits_6dp": float(round(kl_bits, 6)),
    "figure": "parte3_comparacion_barras.png",
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("Distribución renormalizada top-p (T = 1, p = 0.9) y comprobación empírica:")
for t, pt, fo, c in zip(nucleus_tokens, renorm, obs_freq, counts):
    print(f"  {t}: p_teorica = {pt:.6f} | frec_observada = {fo:.6f} | conteo = {int(c)}")
print(f"\nTokens supervivientes: {[str(t) for t in nucleus_tokens]}")
print(f"Tokens excluidos: {[str(t) for t in excluded_tokens]}")
print(f"Probabilidad acumulada del núcleo: {cumulative[k - 1]:.6f}")
print(f"KL(empírica || teórica) = {kl_bits:.6f} bits "
      f"(≈ {kl_bits:.2e} bits, pequeña como se espera con {n_samples} muestras)")
print("\nResultados JSON:")
print(json.dumps(results, indent=2))


Distribución renormalizada top-p (T = 1, p = 0.9) y comprobación empírica:
  t0: p_teorica = 0.569376 | frec_observada = 0.568200 | conteo = 5682
  t1: p_teorica = 0.209462 | frec_observada = 0.209300 | conteo = 2093
  t2: p_teorica = 0.127045 | frec_observada = 0.129600 | conteo = 1296
  t3: p_teorica = 0.094117 | frec_observada = 0.092900 | conteo = 929

Tokens supervivientes: ['t0', 't1', 't2', 't3']
Tokens excluidos: ['t4', 't5']
Probabilidad acumulada del núcleo: 0.968820
KL(empírica || teórica) = 0.000050 bits (≈ 4.99e-05 bits, pequeña como se espera con 10000 muestras)

Resultados JSON:
{
  "logits": [
    2.0,
    1.0,
    0.5,
    0.2,
    -1.0,
    -3.0
  ],
  "temperature": 1.0,
  "p_threshold": 0.9,
  "softmax_T1_full": {
    "t0": 0.5516226378718239,
    "t1": 0.2029306277578035,
    "t2": 0.12308364752983937,
    "t3": 0.09118260875807133,
    "t4": 0.02746367398498595,
    "t5": 0.003716804097476065
  },
  "sorted_order_desc": [
    "t0",
    "t1",
    "t2",
    "t3",
  

## Parte 4 — Pregunta conceptual

**Efecto de la temperatura.** En la softmax con temperatura cada logit se divide por T antes de exponenciar, de modo que el ratio entre las probabilidades de dos tokens es p_i/p_j = exp((z_i − z_j)/T). La temperatura controla, por tanto, cuánto se amplifican o se aplastan las diferencias entre logits.

**Cuando T → 0.** Las diferencias (z_i − z_j)/T crecen sin límite en magnitud: el token con mayor logit (t0 en nuestro vocabulario) absorbe toda la masa de probabilidad y los demás tienden a probabilidad 0. La distribución converge a un delta de Dirac sobre el token más probable, es decir, se vuelve totalmente determinista.

**Por qué T → 0 equivale a la decodificación voraz (greedy).** La decodificación greedy elige en cada paso el token con el mayor logit, sin muestrear. En el límite T → 0, la softmax asigna probabilidad 1 al argmax de los logits y 0 al resto, de modo que muestrear de esa distribución devuelve siempre exactamente ese token: muestrear con T → 0 y tomar el argmax son el mismo procedimiento. (Con T = 0 exacto la fórmula ni siquiera es evaluable por la división entre cero; es el límite el que coincide con greedy. Por eso las APIs ofrecen "temperature = 0" como modo determinista, recomendado por ejemplo para extracción y clasificación.)

**Cuando T → ∞.** En el otro extremo, z_i/T → 0 para todos los tokens, todos los exponenciales tienden a exp(0) = 1 y la distribución converge a la uniforme sobre el vocabulario: todos los tokens pasan a ser igual de probables, incluidos los más improbables, y la generación se vuelve puramente aleatoria y poco coherente.

**Evidencia con las entropías de la Parte 1.** La entropía mide la incertidumbre o "aplanamiento" de la distribución. Como muestra la tabla de la Parte 1, la entropía crece monótonamente con la temperatura: H(T = 0.5) < H(T = 1) < H(T = 2). Bajar T concentra la masa en t0 y reduce la entropía (menos incertidumbre, más determinismo); subir T aplana la distribución y la aumenta (más incertidumbre, más aleatoriedad). Extrapolando esa tendencia: cuando T → 0 la entropía tiende a 0 bits (distribución puntual sobre el token más probable, exactamente greedy) y cuando T → ∞ tiende al máximo posible para un vocabulario de seis tokens, log2(6) bits, correspondiente a la distribución uniforme. Los tres valores calculados en la Parte 1 son los primeros pasos de esa trayectoria entre ambos extremos.